# Temperatura silnika elektrycznego (PMSM) — eksploracyjna analiza danych

### Zrozumienie zbioru danych przed klasteryzacją i augmentacją uwzględniającą klastry

**Rola autora:** Principal Data Scientist — przemysłowa sztuczna inteligencja, analiza szeregów czasowych, uczenie nienadzorowane, generowanie danych syntetycznych, utrzymanie predykcyjne.

**Zbiór danych:** Kaggle — Electric Motor Temperature (wkirgsn), zarejestrowany na stanowisku badawczym silnika synchronicznego z magnesami trwałymi (PMSM) na Uniwersytecie w Paderborn.

**Wykorzystany plik:** `measures_v2.csv`, próbkowany z częstotliwością **2 Hz**.

# 0. Cel notebooka

Jest to **notebook nr 1** w serii notebooków, których długoterminowym celem jest:

> **Augmentacja szeregów czasowych uwzględniająca klastry (Cluster-Aware Time-Series Augmentation)** — najpierw identyfikacja naturalnych reżimów pracy silnika, a następnie generowanie danych syntetycznych z zachowaniem struktury charakterystycznej dla każdego reżimu.

Głównym celem tego notebooka jest **zrozumienie danych, a nie modelowanie**.

W notebooku nie jest budowany żaden model uczenia nadzorowanego. Każdy wniosek jest poparty wartością obliczoną bezpośrednio na podstawie danych lub odpowiednią wizualizacją.

Notebook odpowiada na następujące pytania:

| # | Pytanie |
|---|---|
| 1 | Co zawiera zbiór danych? |
| 2 | Co reprezentuje pojedynczy `profile_id`? |
| 3 | Jak bardzo poszczególne profile różnią się od siebie? |
| 4 | Jakie warunki pracy występują w danych? |
| 5 | Które zmienne wpływają na zachowanie maszyny? |
| 6 | Które zmienne powinny zostać wykorzystane do klasteryzacji? |
| 7 | Których zmiennych nie należy wykorzystywać do klasteryzacji? |
| 8 | Jaka długość segmentu szeregu czasowego może być znacząca? |
| 9 | Czy w danych występują naturalne reżimy pracy? |
| 10 | Czy występują niedostatecznie reprezentowane warunki pracy? |
| 11 | Które podejścia do klasteryzacji wydają się najbardziej obiecujące? |
| 12 | Jakie informacje muszą zostać zachowane podczas augmentacji? |

## Dyscyplina metodologiczna stosowana w całym notebooku

**Wynik statystyczny** — informacja obliczona bezpośrednio na podstawie danych.

**Interpretacja dziedzinowa** — interpretacja uzyskanego wyniku wynikająca z wiedzy o analizowanym układzie.

Te dwa poziomy są zawsze od siebie oddzielane.

Opublikowany zbiór danych został poddany **normalizacji typu z-score przez dostawcę danych**, dlatego zapisane wartości są bezwymiarowe. W notebooku nie przypisuje się znormalizowanym wartościom sztucznych jednostek fizycznych.

# 1. Zrozumienie dziedziny

## 1.1 Czym jest PMSM?

**Silnik synchroniczny z magnesami trwałymi (Permanent Magnet Synchronous Motor, PMSM)** jest trójfazową maszyną prądu przemiennego, której wirnik wyposażony jest w magnesy trwałe.

Silniki PMSM są powszechnie stosowane m.in. w pojazdach elektrycznych, robotyce oraz wysokowydajnych napędach przemysłowych ze względu na dużą gęstość momentu obrotowego oraz wysoką sprawność.

Sterowanie silnikiem odbywa się w wirującym układzie odniesienia **d–q** (sterowanie polowo-zorientowane / wektorowe — FOC). W wyniku transformacji sygnały elektryczne są reprezentowane jako składowe napięć i prądów w osiach `d` i `q`, zamiast jako surowe wartości trzech faz:

- prąd osi **q** (`i_q`, quadrature) jest przede wszystkim związany z generowaniem momentu obrotowego,
- prąd osi **d** (`i_d`, direct) odpowiada za sterowanie strumieniem magnetycznym / osłabianie pola i jest wykorzystywany m.in. podczas pracy z dużymi prędkościami powyżej prędkości bazowej.

# 1. Zrozumienie dziedziny

## 1.1 Czym jest PMSM?

**Silnik synchroniczny z magnesami trwałymi (Permanent Magnet Synchronous Motor, PMSM)** jest trójfazową maszyną prądu przemiennego, której wirnik wyposażony jest w magnesy trwałe.

Silniki PMSM są powszechnie stosowane m.in. w pojazdach elektrycznych, robotyce oraz wysokowydajnych napędach przemysłowych ze względu na dużą gęstość momentu obrotowego oraz wysoką sprawność.

Sterowanie silnikiem odbywa się w wirującym układzie odniesienia **d–q** (sterowanie polowo-zorientowane / wektorowe — FOC). W wyniku transformacji sygnały elektryczne są reprezentowane jako składowe napięć i prądów w osiach `d` i `q`, zamiast jako surowe wartości trzech faz:

- prąd osi **q** (`i_q`, quadrature) jest przede wszystkim związany z generowaniem momentu obrotowego,
- prąd osi **d** (`i_d`, direct) odpowiada za sterowanie strumieniem magnetycznym / osłabianie pola i jest wykorzystywany m.in. podczas pracy z dużymi prędkościami powyżej prędkości bazowej.

## 1.2 Dlaczego monitorowanie temperatury jest istotne?

Temperatura jest jednym z najważniejszych czynników ograniczających trwałość maszyny elektrycznej:

- **magnesy trwałe** mogą ulec nieodwracalnej demagnetyzacji, jeżeli temperatura wirnika stanie się zbyt wysoka, co prowadzi do trwałego pogorszenia zdolności generowania momentu obrotowego,
- **izolacja uzwojeń stojana** starzeje się znacznie szybciej wraz ze wzrostem temperatury, dlatego temperatura uzwojeń ma bezpośredni wpływ na trwałość maszyny,
- **ograniczenia cieplne**, a nie wyłącznie ograniczenia elektryczne, często wyznaczają maksymalny ciągły moment obrotowy i moc możliwe do uzyskania przez silnik.

Wiarygodne monitorowanie stanu cieplnego umożliwia zatem m.in. bezpieczne ograniczanie parametrów pracy, diagnostykę i utrzymanie predykcyjne oraz projektowanie przyspieszonych cykli testowych.

## 1.3 Dlaczego pomiar temperatury PM (wirnika) jest trudny?

Magnesy trwałe znajdują się na obracającym się wirniku. Umieszczenie na nim fizycznego czujnika temperatury wymaga zastosowania np. pierścieni ślizgowych lub systemu telemetrycznego, co znacząco komplikuje pomiar i ogranicza możliwość wykorzystania takiego rozwiązania w seryjnych zastosowaniach.

W analizowanym stanowisku badawczym temperatura `pm` jest dostępna dzięki zastosowaniu specjalistycznego systemu pomiarowego.

Praktycznym celem wykorzystania tego zbioru danych jest jednak możliwość **estymacji temperatury wirnika/magnesów trwałych na podstawie wielkości łatwiejszych do pomiaru**, takich jak napięcia, prądy, prędkość obrotowa czy temperatura chłodziwa i otoczenia.

Rozróżnienie to jest szczególnie istotne w dalszej części analizy: zmienne temperaturowe są **wynikiem odpowiedzi cieplnej układu**, dlatego powinny być zachowane i analizowane, ale nie powinny być automatycznie traktowane jako niezależne zmienne wejściowe opisujące warunki pracy podczas klasteryzacji.

## 1.4 Role zmiennych

| Rola | Zmienne | Co reprezentują |
|---|---|---|
| Zachowanie elektryczne | `u_q`, `u_d`, `i_q`, `i_d` | chwilowe składowe napięć i prądów w układzie d–q |
| Warunki pracy | `motor_speed`, `torque` | mechaniczny punkt pracy maszyny |
| Warunki otoczenia | `coolant`, `ambient` | warunki brzegowe wynikające z układu chłodzenia i temperatury otoczenia |
| Odpowiedź cieplna | `pm`, `stator_yoke`, `stator_tooth`, `stator_winding` | temperatury będące skutkiem pracy maszyny |
| Identyfikator | `profile_id` | niezależna sesja / eksperyment pomiarowy |

## 1.5 Słownik danych

Poniższa tabela opisuje znaczenie poszczególnych zmiennych oraz oczekiwane zależności wynikające z wiedzy dziedzinowej.

Kolumny „Oczekiwany wpływ” oraz „Oczekiwana zależność” reprezentują **hipotezy wynikające z wiedzy dziedzinowej**, określone przed analizą danych. W dalszej części notebooka zostaną one zweryfikowane empirycznie.

# 2. Środowisko i konfiguracja

## 2.1 Instalacja

Poniższa komórka została celowo zakomentowana — notebook nie instaluje automatycznie żadnych pakietów.

Pakiet `umap-learn` jest opcjonalny. Jeżeli nie jest dostępny, analiza może zostać przeprowadzona z wykorzystaniem PCA oraz t-SNE.

In [2]:
# ============================================================
# INSTALACJA — uruchomić ręcznie tylko w razie potrzeby
# ============================================================

%pip install pandas numpy matplotlib seaborn scipy scikit-learn umap-learn

# Uwagi:
#   * umap-learn — OPCJONALNY; umożliwia wykorzystanie UMAP.
#     Jeśli pakiet nie jest dostępny, można wykorzystać PCA i t-SNE.
#
#   * psutil — OPCJONALNY; wykorzystywany wyłącznie do raportowania
#     wykorzystania pamięci procesu.

print("Komórka instalacyjna ma charakter informacyjny — nie zainstalowano żadnych pakietów.")


Note: you may need to restart the kernel to use updated packages.
Komórka instalacyjna ma charakter informacyjny — nie zainstalowano żadnych pakietów.


## 2.2 Importy i raport wersji

Zależności opcjonalne są obsługiwane w sposób defensywny — brak opcjonalnego pakietu może ograniczyć dostępność wybranej funkcjonalności, ale nie powinien powodować przerwania działania całego notebooka.

## 2.3 Konfiguracja

In [6]:
# ============================================================
# KONFIGURACJA — jedno źródło ustawień dla całego notebooka
# ============================================================

# --- lokalizacja danych -------------------------------------

DATA_PATH = "../dataset/raw/measures_v2.csv"       # plik CSV Kaggle; ścieżka względna względem notebooka
ID_COL = "profile_id"               # kolumna identyfikująca niezależne szeregi czasowe


# --- grupy cech (role określone w słowniku danych) ----------

ELECTRICAL_FEATURES = ["u_q", "u_d", "i_q", "i_d"]

OPERATIONAL_FEATURES = ["motor_speed", "torque"]

CONTEXT_FEATURES = ["coolant", "ambient"]

TEMPERATURE_FEATURES = [
    "pm",
    "stator_yoke",
    "stator_tooth",
    "stator_winding"
]


# Zbiory pomocnicze wykorzystywane w całym notebooku

DRIVER_FEATURES = ELECTRICAL_FEATURES + OPERATIONAL_FEATURES
# zmienne opisujące sterowanie i bieżący punkt pracy

CONDITION_FEATURES = DRIVER_FEATURES + CONTEXT_FEATURES
# potencjalne zmienne wejściowe do klasteryzacji

ALL_FEATURES = CONDITION_FEATURES + TEMPERATURE_FEATURES
# wszystkie kanały pomiarowe


# --- próbkowanie / segmentacja ------------------------------

SAMPLING_FREQUENCY_HZ = 2           # 2 Hz -> 0,5 s na próbkę
SECONDS_PER_SAMPLE = 1.0 / SAMPLING_FREQUENCY_HZ

SEGMENT_LENGTHS_SECONDS = [60, 120, 300]     # kandydaci: 1, 2 i 5 minut

SEGMENT_LENGTHS = [
    int(s * SAMPLING_FREQUENCY_HZ)
    for s in SEGMENT_LENGTHS_SECONDS
]

SEGMENT_OVERLAP = 0.5               # 50% nakładania się kolejnych okien


# --- powtarzalność / wejście-wyjście ------------------------

RANDOM_STATE = 42
OUTPUT_DIRECTORY = "eda_outputs"


# --- ograniczenia dotyczące wykresów i próbkowania ----------

N_EXAMPLE_PROFILES_TO_PLOT = 3
# liczba reprezentatywnych profili prezentowanych na wykresach czasowych

MAX_POINTS_FOR_SCATTER = 40_000
# maksymalna liczba losowo wybranych punktów na wykresach scatter/hexbin

MAX_ROWS_FOR_PAIRPLOT = 4_000
# ograniczenie liczby obserwacji dla kosztownego obliczeniowo pairplot


# --- mechanizm awaryjny danych syntetycznych ----------------

USE_SYNTHETIC_DATA = False
# Domyślnie False.
# Dane syntetyczne służą wyłącznie do demonstracji działania notebooka offline.

N_SYNTHETIC_PROFILES = 12
SYNTHETIC_MIN_LENGTH = 1_600
SYNTHETIC_MAX_LENGTH = 4_200

In [7]:
CONFIG_SNAPSHOT: dict[str, Any] = {
    "DATA_PATH": DATA_PATH,
    "ID_COL": ID_COL,
    "ELECTRICAL_FEATURES": list(ELECTRICAL_FEATURES),
    "OPERATIONAL_FEATURES": list(OPERATIONAL_FEATURES),
    "CONTEXT_FEATURES": list(CONTEXT_FEATURES),
    "TEMPERATURE_FEATURES": list(TEMPERATURE_FEATURES),
    "SAMPLING_FREQUENCY_HZ": SAMPLING_FREQUENCY_HZ,
    "SEGMENT_LENGTHS_SECONDS": list(SEGMENT_LENGTHS_SECONDS),
    "SEGMENT_LENGTHS": list(SEGMENT_LENGTHS),
    "RANDOM_STATE": RANDOM_STATE,
    "OUTPUT_DIRECTORY": OUTPUT_DIRECTORY,
    "USE_SYNTHETIC_DATA": USE_SYNTHETIC_DATA,
}

print(f"Cechy elektryczne: {ELECTRICAL_FEATURES}")
print(f"Cechy operacyjne: {OPERATIONAL_FEATURES}")
print(f"Cechy kontekstowe: {CONTEXT_FEATURES}")
print(f"Zmienne temperaturowe: {TEMPERATURE_FEATURES}")
print(
    f"Kandydaci długości segmentów: {SEGMENT_LENGTHS_SECONDS} s "
    f"-> {SEGMENT_LENGTHS} próbek przy {SAMPLING_FREQUENCY_HZ} Hz"
)

NameError: name 'Any' is not defined

## 2.4 Logowanie, katalog wynikowy i funkcje pomocnicze

Wszystkie wygenerowane tabele CSV i wykresy są zapisywane w jednym katalogu wynikowym. Niewielkie, dobrze udokumentowane funkcje pomocnicze pozwalają zachować przejrzystość komórek odpowiedzialnych za właściwą analizę danych.

In [ ]:
class Timer:
    """Minimalny menedżer kontekstu mierzący czas wykonania bloku kodu."""

def current_memory_mb() -> float:
    """Zwraca wykorzystanie pamięci RAM bieżącego procesu w MiB.
    Jeśli pakiet psutil nie jest dostępny, zwracana jest wartość NaN.
    """

def save_figure(fig: plt.Figure, name: str) -> None:
    """Zapisuje wykres do katalogu wynikowego.
    Błąd zapisu wykresu nie powinien przerywać całej analizy.
    """

def export_table(
    frame: pd.DataFrame,
    filename: str,
    description: str,
    index: bool = False
) -> pd.DataFrame:
    """Zapisuje tabelę do pliku CSV i rejestruje ją wśród wyników analizy."""

def subsample_rows(frame: pd.DataFrame, max_rows: int) -> pd.DataFrame:
    """Zwraca powtarzalną losową próbkę danych, jeżeli liczba wierszy
    przekracza wartość `max_rows`.
    """

# 3. Wczytywanie danych

Notebook oczekuje pliku Kaggle `measures_v2.csv` wskazanego przez zmienną `DATA_PATH`.

Jeżeli plik nie zostanie znaleziony, notebook zgłasza jednoznaczny błąd i zatrzymuje wykonywanie. Dane zastępcze nie są generowane automatycznie.

Opcjonalny mechanizm `USE_SYNTHETIC_DATA` istnieje wyłącznie w celu demonstracji działania całego pipeline'u w środowisku, w którym oryginalny zbiór danych nie jest dostępny.

Wyniki uzyskane na danych syntetycznych **nie mogą być interpretowane jako wyniki dotyczące rzeczywistego silnika PMSM**.

## Jak pozyskać dane

1. Otwórz stronę zbioru **Electric Motor Temperature** na Kaggle.
2. Pobierz archiwum i rozpakuj plik `measures_v2.csv`.
3. Umieść plik obok notebooka lub ustaw odpowiednią ścieżkę w zmiennej `DATA_PATH`.

In [ ]:
def make_synthetic_motor_data(
    n_profiles: int,
    min_length: int,
    max_length: int,
    id_col: str,
    random_state: int,
) -> pd.DataFrame:
    """
    Generuje jednoznacznie oznaczony syntetyczny odpowiednik zbioru PMSM.

    Generator odtwarza oczekiwane nazwy kolumn, kilka wartości `profile_id`
    o różnych długościach oraz cztery rozróżnialne archetypy pracy:
    małe obciążenie w stanie ustalonym, duża prędkość i duże obciążenie,
    pracę przejściową/cykliczną oraz narastanie i opadanie obciążenia.

    Funkcja istnieje wyłącznie po to, aby umożliwić demonstracyjne
    uruchomienie notebooka offline.

    Należy ją usunąć lub pozostawić wyłączoną, gdy dostępny jest
    rzeczywisty plik z Kaggle.
    """

def load_motor_data(
    data_path: str,
    id_col: str,
    use_synthetic: bool
) -> tuple[pd.DataFrame, bool]:
    """Wczytuje pomiary PMSM lub zgłasza jednoznaczny błąd."""

if not use_synthetic:
    raise FileNotFoundError(
        "\n" + "=" * 78 +
        f"\nNIE ZNALEZIONO PLIKU Z DANYMI: {path.resolve()}\n"
        "\nTen notebook wymaga zbioru Kaggle „Electric Motor Temperature”.\n"
        "\nJak rozwiązać problem:\n"
        "  1. Pobierz zbiór Electric Motor Temperature z Kaggle.\n"
        "  2. Rozpakuj plik measures_v2.csv.\n"
        f"  3. Umieść go w: {path.resolve()}\n"
        "     (lub zmień DATA_PATH w komórce konfiguracyjnej).\n"
        "\nDane zastępcze NIE są generowane automatycznie.\n"
        "Aby uruchomić notebook wyłącznie demonstracyjnie, ustaw\n"
        "USE_SYNTHETIC_DATA = True.\n"
        "Wyniki uzyskane wtedy na danych syntetycznych NIE są wynikami "
        "dotyczącymi rzeczywistego silnika.\n"
    )


if DATA_IS_SYNTHETIC:
    print("!" * 78)
    print("!! UWAGA: notebook działa na DANYCH SYNTETYCZNYCH.")
    print("!! Wszystkie poniższe wykresy i tabele służą wyłącznie demonstracji.")
    print("!! Nie należy na ich podstawie wyciągać wniosków dotyczących rzeczywistego silnika.")
    print("!" * 78)

missing = [c for c in ALL_FEATURES + [ID_COL] if c not in raw_df.columns]

if missing:
    raise KeyError(
        f"W zbiorze brakuje wymaganych kolumn: {missing}\n"
        f"Dostępne kolumny: {list(raw_df.columns)}"
    )

print(f"Źródło danych: {DATA_LABEL}")
print(f"Rozmiar: {raw_df.shape[0]:,} wierszy × {raw_df.shape[1]} kolumn")

raw_df.head()

## 3.1 Zapis słownika danych

Informacje dziedzinowe z sekcji 1.5 są zapisywane do pliku `data_dictionary.csv`, dzięki czemu kolejne notebooki korzystają z jednego, spójnego opisu wszystkich zmiennych.

# 4. Przegląd zbioru danych i jakość danych

Przed rozpoczęciem interpretacji przeprowadzany jest audyt surowego zbioru danych obejmujący:

- liczbę obserwacji i zmiennych,
- wykorzystanie pamięci,
- brakujące wartości,
- duplikaty,
- typy danych,
- integralność kolumny identyfikującej profile.

Wyniki audytu są zapisywane w tabelach wynikowych, aby zapewnić możliwość późniejszego odtworzenia i udokumentowania analizy.

In [ ]:
n_rows, n_cols = raw_df.shape
mem_bytes = raw_df.memory_usage(deep=True).sum()
total_duration_hours = n_rows * SECONDS_PER_SAMPLE / 3600.0

overview = pd.DataFrame(
    {
        "metryka": [
            "liczba wierszy",
            "liczba kolumn",
            "liczba profili",
            "pamięć (MB)",
            "częstotliwość próbkowania (Hz)",
            "sekundy na próbkę",
            "całkowity czas rejestracji (godz.)",
            "źródło danych",
        ],
        "wartość": [
            f"{n_rows:,}",
            f"{n_cols}",
            f"{raw_df[ID_COL].nunique():,}",
            f"{mem_bytes / 1024**2:,.2f}",
            f"{SAMPLING_FREQUENCY_HZ}",
            f"{SECONDS_PER_SAMPLE}",
            f"{total_duration_hours:,.2f}",
            DATA_LABEL,
        ],
    }
)

print("Podsumowanie zbioru danych")
print("=" * 60)

overview

### 4.1 Kontrola jakości poszczególnych kolumn i walidacja typów danych

Dla każdej kolumny raportowany jest typ danych, liczba wartości niepustych, udział brakujących danych, liczba wartości nieskończonych, liczba wartości unikalnych oraz informacja, czy dana kolumna jest (nieoczekiwanie) stała.

Dla kolumny `profile_id` dodatkowo sprawdzana jest jej poprawność, w szczególności to, czy identyfikatory profili mają postać dodatnich liczb całkowitych.

In [ ]:
def build_quality_report(frame: pd.DataFrame) -> pd.DataFrame:
    records = []

    for col in frame.columns:
        series = frame[col]
        is_numeric = pd.api.types.is_numeric_dtype(series)
        n_inf = int(np.isinf(series.to_numpy()).sum()) if is_numeric else 0
        nunique = int(series.nunique(dropna=True))

        records.append(
            {
                "column": col,
                "dtype": str(series.dtype),
                "non_null": int(series.notna().sum()),
                "missing": int(series.isna().sum()),
                "missing_pct": 100.0 * series.isna().mean(),
                "n_infinite": n_inf,
                "n_unique": nunique,
                "is_constant": nunique <= 1,
                "mean": series.mean() if is_numeric else np.nan,
                "std": series.std() if is_numeric else np.nan,
            }
        )

    return pd.DataFrame(records)


quality_report = build_quality_report(raw_df)

export_table(
    quality_report,
    "data_quality_report.csv",
    "Audyt jakości danych i typów dla poszczególnych kolumn"
)

quality_report

### 4.2 Duplikaty, integralność identyfikatorów i walidacja globalna

Sprawdzamy występowanie zduplikowanych wierszy, poprawność kolumny `profile_id` oraz potwierdzamy brak wartości nieskończonych i nieoczekiwanych stałych kanałów pomiarowych.

In [ ]:
n_duplicate_rows = int(raw_df.duplicated().sum())

id_series = raw_df[ID_COL]

id_is_integer_like = bool(
    np.allclose(id_series.dropna() % 1, 0)
)

id_has_missing = bool(id_series.isna().any())
id_min = id_series.min()

n_infinite_total = int(quality_report["n_infinite"].sum())

constant_sensor_cols = [
    c for c in ALL_FEATURES
    if c in quality_report.set_index("column").index
    and bool(quality_report.set_index("column").loc[c, "is_constant"])
]

missing_sensor_cols = [
    c for c in ALL_FEATURES
    if int(
        quality_report.set_index("column").loc[c, "missing"]
    ) > 0
]

validation = pd.DataFrame(
    {
        "check": [
            "zduplikowane wiersze",
            "profile_id ma wartości całkowite",
            "brakujące profile_id",
            "minimalne profile_id",
            "łączna liczba wartości nieskończonych",
            "stałe kanały pomiarowe",
            "kanały pomiarowe z brakującymi wartościami",
        ],
        "result": [
            f"{n_duplicate_rows}",
            str(id_is_integer_like),
            str(id_has_missing),
            f"{id_min}",
            f"{n_infinite_total}",
            ", ".join(constant_sensor_cols) if constant_sensor_cols else "brak",
            ", ".join(missing_sensor_cols) if missing_sensor_cols else "brak",
        ],
        "status": [
            "OK" if n_duplicate_rows == 0 else "SPRAWDŹ",
            "OK" if id_is_integer_like else "SPRAWDŹ",
            "OK" if not id_has_missing else "SPRAWDŹ",
            "OK" if (pd.notna(id_min) and id_min >= 0) else "SPRAWDŹ",
            "OK" if n_infinite_total == 0 else "SPRAWDŹ",
            "OK" if not constant_sensor_cols else "SPRAWDŹ",
            "OK" if not missing_sensor_cols else "SPRAWDŹ",
        ],
    }
)

print("Podsumowanie walidacji")
print("-" * 60)

validation

### 4.3 Profil wykorzystania pamięci

Analizujemy wykorzystanie pamięci przez poszczególne kolumny. Informacja ta jest przydatna przy planowaniu bardziej rozbudowanych etapów klasteryzacji i augmentacji, podczas których wiele profili może być jednocześnie przechowywanych w pamięci.

In [ ]:
mem_profile = (
    raw_df.memory_usage(deep=True)
    .drop(labels="Index", errors="ignore")
    .sort_values(ascending=False)
    .rename("bytes")
    .to_frame()
)

mem_profile["MB"] = mem_profile["bytes"] / 1024**2
mem_profile["pct"] = 100.0 * mem_profile["bytes"] / mem_profile["bytes"].sum()

fig, ax = plt.subplots(figsize=(9, 5))

ax.barh(
    mem_profile.index[::-1],
    mem_profile["MB"][::-1],
    color=sns.color_palette("crest", len(mem_profile))
)

ax.set_xlabel("Pamięć [MB]")
ax.set_title(f"Profil wykorzystania pamięci według kolumn | {DATA_LABEL}")

fig.tight_layout()
save_figure(fig, "04_memory_profile")
plt.show()

mem_profile[["MB", "pct"]]

> **Wnioski statystyczne (przegląd zbioru danych).**  
> Powyższy audyt przedstawia dokładną liczbę wierszy i kolumn, liczbę niezależnych profili, całkowity czas rejestracji przy znanej częstotliwości próbkowania 2 Hz oraz kontrolę integralności poszczególnych kolumn.
>
> **Interpretacja dziedzinowa.**  
> Dla oryginalnego zbioru danych oczekujemy braku wartości brakujących i nieskończonych, braku problematycznych duplikatów, poprawnych dodatnich identyfikatorów `profile_id` oraz braku stałych kanałów pomiarowych. Każdy status `SPRAWDŹ` powinien zostać przeanalizowany przed rozpoczęciem klasteryzacji — stały kanał nie wnosi informacji różnicującej, natomiast braki danych mogą zaburzać obliczenia odległości.

## 5. Analiza profili

Każdy `profile_id` reprezentuje niezależny eksperyment — samodzielną sesję pomiarową o określonym czasie trwania.

Kolejność wierszy wewnątrz profilu odpowiada kolejności czasowej. Ponieważ zbiór danych nie zawiera jawnej kolumny ze znacznikiem czasu, oś czasu wyznaczamy na podstawie indeksu próbki i znanej częstotliwości próbkowania wynoszącej 2 Hz.

Profile **nie mogą być łączone ani przecinać się** podczas późniejszego tworzenia segmentów.

W tej części odpowiadamy na pytania: **czym jest pojedynczy profil, ile profili znajduje się w zbiorze, jak długie są profile oraz jak bardzo różnią się między sobą?**

In [ ]:
# Dodanie jawnej osi czasu dla każdego profilu
# (czas w sekundach od początku danej rejestracji)

work_df = raw_df.copy()

work_df["_row_in_profile"] = work_df.groupby(ID_COL).cumcount()

work_df["_time_seconds"] = (
    work_df["_row_in_profile"] * SECONDS_PER_SAMPLE
)

profile_ids = sorted(work_df[ID_COL].unique())
n_profiles = len(profile_ids)

grouped = work_df.groupby(ID_COL)

print(f"Liczba profili: {n_profiles}")
print(
    f"Zakres identyfikatorów profile_id: "
    f"{min(profile_ids)} ... {max(profile_ids)}"
)

### 5.1 Tabela podsumowująca profile

Każdy wiersz tabeli odpowiada jednemu profilowi i podsumowuje jego wielkość, czas trwania oraz zachowanie operacyjne i termiczne obserwowane podczas sesji.

Zmienna `temperature_increase` wykorzystuje średnią z czterech kanałów temperaturowych — różnicę pomiędzy początkiem i końcem profilu — jako zwięzłą miarę tego, jak bardzo maszyna nagrzała się podczas sesji.

Tabela jest eksportowana do pliku `profile_summary.csv`.

In [ ]:
def summarise_profile(pid: int, frame: pd.DataFrame) -> dict[str, Any]:
    n = len(frame)
    duration_s = n * SECONDS_PER_SAMPLE

    temp_mean_series = frame[TEMPERATURE_FEATURES].mean(axis=1)

    # Średnia temperatura z pierwszego i ostatniego 5% profilu
    # – rozwiązanie bardziej odporne na szum.
    edge = max(1, n // 20)

    temp_start = temp_mean_series.iloc[:edge].mean()
    temp_end = temp_mean_series.iloc[-edge:].mean()

    return {
        ID_COL: pid,
        "samples": n,
        "duration_s": duration_s,
        "duration_min": duration_s / 60.0,
        "avg_speed": frame["motor_speed"].mean(),
        "avg_torque": frame["torque"].mean(),
        "avg_temperature": temp_mean_series.mean(),
        "max_temperature": temp_mean_series.max(),
        "temperature_increase": temp_end - temp_start,
        "speed_std": frame["motor_speed"].std(),
        "torque_std": frame["torque"].std(),

        # Zwięzła miara dynamiki profilu:
        # średnie odchylenie standardowe cech sterujących.
        "dynamics_score": frame[DRIVER_FEATURES].std().mean(),
    }


profile_summary = pd.DataFrame(
    [
        summarise_profile(pid, grouped.get_group(pid))
        for pid in profile_ids
    ]
)

export_table(
    profile_summary,
    "profile_summary.csv",
    "Podsumowanie wielkości, czasu trwania i zachowania profili"
)

# Identyfikacja charakterystycznych profili
# wykorzystywanych w dalszej analizie.

shortest_profile = int(
    profile_summary.loc[
        profile_summary["samples"].idxmin(), ID_COL
    ]
)

longest_profile = int(
    profile_summary.loc[
        profile_summary["samples"].idxmax(), ID_COL
    ]
)

most_variable_profile = int(
    profile_summary.loc[
        profile_summary["dynamics_score"].idxmax(), ID_COL
    ]
)

least_variable_profile = int(
    profile_summary.loc[
        profile_summary["dynamics_score"].idxmin(), ID_COL
    ]
)

hottest_profile = int(
    profile_summary.loc[
        profile_summary["max_temperature"].idxmax(), ID_COL
    ]
)

print(
    f"Najkrótszy profil: {shortest_profile} "
    f"({profile_summary['samples'].min():,} próbek)"
)

print(
    f"Najdłuższy profil: {longest_profile} "
    f"({profile_summary['samples'].max():,} próbek)"
)

print(
    f"Najbardziej zmienny profil: {most_variable_profile}"
)

print(
    f"Najmniej zmienny profil: {least_variable_profile}"
)

print(
    f"Profil o najwyższej temperaturze: {hottest_profile}"
)

profile_summary.sort_values(
    "samples",
    ascending=False
).head(12)

### 5.2 Rozkład długości i czasu trwania profili

Poniższe dwa histogramy pokazują, jak nierównomierne są długości poszczególnych profili.

Silna prawostronna skośność — niewielka liczba bardzo długich sesji oraz wiele krótszych — jest istotną właściwością strukturalną zbioru danych. Uzasadnia ona późniejsze prowadzenie analizy na dwóch poziomach: dla całych profili po normalizacji długości oraz dla segmentów o stałej długości.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(
    profile_summary["samples"],
    bins=min(30, n_profiles),
    color="#4C72B0",
    edgecolor="white"
)

axes[0].axvline(
    profile_summary["samples"].median(),
    color="crimson",
    ls="--",
    label="mediana"
)

axes[0].set(
    xlabel="Liczba próbek w profilu",
    ylabel="Liczba profili",
    title="Rozkład długości profili"
)

axes[0].legend()

axes[1].hist(
    profile_summary["duration_min"],
    bins=min(30, n_profiles),
    color="#55A868",
    edgecolor="white"
)

axes[1].axvline(
    profile_summary["duration_min"].median(),
    color="crimson",
    ls="--",
    label="mediana"
)

axes[1].set(
    xlabel="Czas trwania [min]",
    ylabel="Liczba profili",
    title="Rozkład czasu trwania profili"
)

axes[1].legend()

fig.suptitle(
    f"Zróżnicowanie długości profili | {DATA_LABEL}"
)

fig.tight_layout()

save_figure(fig, "05_profile_length_duration_hist")
plt.show()

length_stats = profile_summary["samples"].describe()

print("Długość profilu [liczba próbek]:")
print(length_stats.to_string())

print(
    f"\nSkośność długości: "
    f"{profile_summary['samples'].skew():.3f}"
)

print(
    f"Stosunek najdłuższego do najkrótszego profilu: "
    f"{profile_summary['samples'].max() / max(profile_summary['samples'].min(), 1):.1f}x"
)

### 5.3 Jak bardzo różnią się profile? Charakterystyka punktów pracy

Każdy profil redukujemy do jego średniego punktu pracy — średniej prędkości obrotowej oraz średniego momentu obrotowego — a kolor punktu reprezentuje stopień nagrzania maszyny.

Duży rozrzut punktów oznacza, że poszczególne profile rzeczywiście obejmują odmienne obszary pracy silnika. Jest to materiał, który w dalszej części analizy będzie grupowany za pomocą metod klasteryzacji.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sc = axes[0].scatter(
    profile_summary["avg_speed"],
    profile_summary["avg_torque"],
    c=profile_summary["temperature_increase"],
    cmap="magma",
    s=90,
    edgecolor="k",
    linewidth=0.5,
)

for _, r in profile_summary.iterrows():
    axes[0].annotate(
        int(r[ID_COL]),
        (r["avg_speed"], r["avg_torque"]),
        fontsize=7,
        alpha=0.7,
    )

axes[0].set(
    xlabel="Średnia prędkość obrotowa",
    ylabel="Średni moment obrotowy",
    title="Średni punkt pracy dla każdego profilu",
)

fig.colorbar(
    sc,
    ax=axes[0],
    label="temperature_increase"
)

axes[1].scatter(
    profile_summary["dynamics_score"],
    profile_summary["max_temperature"],
    color="tab:gray",
    s=90,
    edgecolor="k",
    linewidth=0.5,
)

axes[1].set(
    xlabel="dynamics_score (średnie odchylenie std. cech sterujących)",
    ylabel="Maksymalna temperatura",
    title="Profile dynamiczne a stabilne",
)

fig.suptitle(
    f"Zróżnicowanie profili | {DATA_LABEL}"
)

fig.tight_layout()
plt.show()

> **Wnioski statystyczne.**  
> Powyżej przedstawiono liczbę profili, rozkład ich długości wraz ze skośnością i stosunkiem długości najdłuższego do najkrótszego profilu, a także ich rozmieszczenie w przestrzeni prędkość–moment.
>
> **Interpretacja dziedzinowa.**  
> Profile są z założenia heterogeniczne — każdy z nich reprezentuje celowo odmienny cykl pracy. Znaczne różnice długości powodują, że bezpośrednie porównywanie całych profili o surowej długości za pomocą odległości euklidesowej nie jest właściwe. Profile powinny zostać znormalizowane względem długości lub podzielone na segmenty.
>
> Rozrzut średnich punktów pracy stanowi pierwszą bezpośrednią przesłankę wskazującą na występowanie różnych reżimów pracy silnika — dokładnie takich struktur, które powinny zostać wykryte przez dalszą klasteryzację.

## 6. Eksploracja cech — rozkłady jednowymiarowe

Dla każdego kanału pomiarowego wyznaczamy pełny zestaw statystyk opisowych, obejmujący m.in. skośność, kurtozę oraz liczbę obserwacji odstających wyznaczoną na podstawie rozstępu międzykwartylowego (IQR).

Dodatkowo analizujemy cztery uzupełniające się sposoby prezentacji rozkładów: histogramy, estymację gęstości KDE, wykresy pudełkowe oraz wykresy skrzypcowe.

Rozkłady analizowane są dla danych połączonych ze wszystkich profili, natomiast struktura samych profili została przeanalizowana oddzielnie w sekcji 5.

In [ ]:
def descriptive_stats(
    frame: pd.DataFrame,
    cols: Sequence[str]
) -> pd.DataFrame:

    records = []

    for col in cols:
        s = frame[col].dropna()

        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1

        lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr

        n_out = int(
            ((s < lo) | (s > hi)).sum()
        )

        records.append(
            {
                "feature": col,
                "mean": s.mean(),
                "median": s.median(),
                "std": s.std(),
                "min": s.min(),
                "q1": q1,
                "q3": q3,
                "max": s.max(),
                "iqr": iqr,
                "skewness": s.skew(),
                # Kurtoza nadmiarowa Fishera
                "kurtosis": s.kurtosis(),
                "n_outliers_iqr": n_out,
                "outlier_pct": 100.0 * n_out / max(len(s), 1),
                "range": s.max() - s.min(),
            }
        )

    return pd.DataFrame(records)


feature_statistics = descriptive_stats(
    work_df,
    ALL_FEATURES
)

feature_statistics.insert(
    1,
    "category",
    feature_statistics["feature"].map(
        dict(
            zip(
                data_dictionary["variable"],
                data_dictionary["category"]
            )
        )
    )
)

export_table(
    feature_statistics,
    "feature_statistics.csv",
    "Statystyki opisowe dla każdego kanału pomiarowego"
)

feature_statistics

### 6.1 Histogramy i estymacja gęstości KDE

Dla każdej cechy tworzony jest osobny panel. Na histogram nakładana jest krzywa estymacji gęstości KDE.

Widoczna wielomodalność rozkładu może wskazywać na występowanie odmiennych reżimów pracy silnika i dlatego będzie istotną przesłanką podczas późniejszej klasteryzacji.

In [ ]:
n = len(ALL_FEATURES)
ncols = 3
nrows = int(np.ceil(n / ncols))

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(15, 3.4 * nrows)
)

axes = np.atleast_1d(axes).ravel()

for ax, col in zip(axes, ALL_FEATURES):
    sample = subsample_rows(
        work_df[[col]],
        MAX_POINTS_FOR_SCATTER
    )[col].dropna()

    sns.histplot(
        sample,
        kde=True,
        ax=ax,
        color="#4C72B0",
        edgecolor="white",
        stat="density"
    )

    ax.set_title(col)
    ax.set_xlabel("")

for ax in axes[n:]:
    ax.set_visible(False)

fig.suptitle(
    f"Rozkłady cech z nałożoną estymacją KDE | {DATA_LABEL}",
    y=1.002
)

fig.tight_layout()

save_figure(fig, "06_histograms_kde")
plt.show()

### 6.2 Wykresy pudełkowe i skrzypcowe

Zmienne grupowane są według pełnionej roli. Wyłącznie na potrzeby wizualizacji wartości są standaryzowane osobno dla każdej cechy, dzięki czemu zmienne występujące naturalnie w różnych skalach mogą być bezpośrednio porównywane.

Dane źródłowe pozostają niezmienione.

Wykresy pudełkowe ułatwiają identyfikację wartości odstających, natomiast wykresy skrzypcowe pozwalają lepiej ocenić kształt i potencjalną wielomodalność rozkładów.

In [ ]:
def zscore_for_display(
    frame: pd.DataFrame,
    cols: Sequence[str]
) -> pd.DataFrame:

    sub = subsample_rows(
        frame[list(cols)],
        MAX_POINTS_FOR_SCATTER
    )

    z = (sub - sub.mean()) / sub.std(ddof=0)

    return z.melt(
        var_name="feature",
        value_name="z"
    )


groups = {
    "elektryczne": ELECTRICAL_FEATURES,
    "operacyjne": OPERATIONAL_FEATURES,
    "kontekstowe": CONTEXT_FEATURES,
    "termiczne": TEMPERATURE_FEATURES,
}

fig, axes = plt.subplots(
    2,
    2,
    figsize=(15, 10)
)

for ax, (name, cols) in zip(axes.ravel(), groups.items()):
    melted = zscore_for_display(work_df, cols)

    sns.violinplot(
        data=melted,
        x="feature",
        y="z",
        ax=ax,
        inner=None,
        cut=0,
        color="#DDDDDD"
    )

    sns.boxplot(
        data=melted,
        x="feature",
        y="z",
        ax=ax,
        width=0.25,
        showfliers=True,
        boxprops={"zorder": 2, "facecolor": "#4C72B0"},
        flierprops={"markersize": 1}
    )

    ax.set_title(
        f"Cechy {name} (standaryzowany z-score)"
    )

    ax.set_xlabel("")
    ax.set_ylabel("z-score")

fig.suptitle(
    f"Wykresy pudełkowe i skrzypcowe według roli | {DATA_LABEL}",
    y=1.002
)

### 6.3 Automatyczne oznaczanie: skośność, ciężkie ogony, obserwacje odstające i potencjalna wielomodalność

Prosty zestaw reguł przekształca tabelę statystyk w łatwe do interpretacji oznaczenia charakterystyki rozkładów.

Należy pamiętać, że są to **wstępne wskaźniki diagnostyczne, a nie ostateczne wnioski statystyczne**.

In [ ]:
def flag_distribution(row: pd.Series) -> str:
    flags = []

    if abs(row["skewness"]) > 1.0:
        flags.append("silna skośność")
    elif abs(row["skewness"]) > 0.5:
        flags.append("umiarkowana skośność")

    if row["kurtosis"] > 3.0:
        flags.append("ciężkie ogony")

    elif row["kurtosis"] < -1.0:
        flags.append("spłaszczony rozkład")

    if row["outlier_pct"] > 5.0:
        flags.append("wiele obserwacji odstających wg IQR")

    # Bardzo płaski rozkład (ujemna kurtoza) lub rozkład
    # z plateau może być sygnałem potencjalnej wielomodalności.
    if row["kurtosis"] < -0.8:
        flags.append("możliwa wielomodalność")

    return "; ".join(flags) if flags else "w przybliżeniu jednomodalny / symetryczny"


dist_flags = feature_statistics[
    [
        "feature",
        "category",
        "skewness",
        "kurtosis",
        "outlier_pct"
    ]
].copy()

dist_flags["flags"] = feature_statistics.apply(
    flag_distribution,
    axis=1
)

print("Wstępna diagnostyka rozkładów")
print("-" * 70)

dist_flags

> **Wnioski statystyczne.**  
> Plik `feature_statistics.csv` zawiera średnią, medianę, odchylenie standardowe, wartości minimalne i maksymalne, kwartyle, IQR, skośność, kurtozę nadmiarową oraz liczbę obserwacji odstających według kryterium IQR dla wszystkich kanałów. Tabela flag podsumowuje nietypowe właściwości rozkładów.
>
> **Interpretacja dziedzinowa.**
>
> - `motor_speed` oraz `torque` należą do zmiennych, w których najbardziej prawdopodobna jest **wielomodalność**. Cykle pracy celowo obejmują określone punkty pracy — np. bieg jałowy, jazdę ze stałą prędkością czy duże obciążenie — co może prowadzić do powstawania wielu maksimów rozkładu. Jest to korzystna informacja z punktu widzenia późniejszej klasteryzacji.
> - Kanały temperaturowe są zazwyczaj **prawostronnie skośne**: maszyna przez znaczną część czasu pracuje w pobliżu niższych temperatur, a okresowo osiąga wyższe wartości. Obserwacje odstające dla temperatur są więc fizycznie możliwymi wysokimi temperaturami, a nie wartościami, które należy automatycznie usuwać.
> - `ambient` i `coolant` są zmiennymi wolnozmiennymi o niewielkim zakresie. Duża liczba obserwacji odstających według IQR może w ich przypadku odzwierciedlać kilka różnych warunków eksperymentalnych, a nie szum pomiarowy.

## 7. Eksploracja szeregów czasowych

Analiza rozkładów jednowymiarowych pomija informację o czasie. W tej części analizujemy reprezentatywne profile jako **wielowymiarowe trajektorie czasowe**.

Dla poszczególnych grup cech tworzymy oddzielne panele, aby zobaczyć, w jaki sposób maszyna przemieszcza się w przestrzeni stanów pracy: przyspieszenia i hamowania, stabilne poziomy pracy, przejścia pomiędzy stanami oraz opóźnioną względem nich odpowiedź termiczną.

Reprezentatywne profile dobrano tak, aby obejmowały zakres zachowań zidentyfikowany w sekcji 5 — m.in. profil najdłuższy, najbardziej zmienny, najgorętszy i najmniej zmienny. Powtarzające się profile są usuwane z zestawu.

In [ ]:
representative_profiles = list(
    dict.fromkeys(
        [
            most_variable_profile,
            longest_profile,
            hottest_profile,
            least_variable_profile,
        ]
    )
)[:MAX_N_EXAMPLE_PROFILES_TO_PLOT]

print(
    "Reprezentatywne profile wybrane do analizy szeregów czasowych:",
    representative_profiles
)


def plot_profile_timeseries(pid: int) -> None:
    frame = grouped.get_group(pid).reset_index(drop=True)

    t_min = frame["_time_seconds"] / 60.0

    panels = [
        ("motor_speed", OPERATIONAL_FEATURES[:1], "operacyjne"),
        ("torque", OPERATIONAL_FEATURES[1:], "operacyjne"),
        ("prądy i_q / i_d", ["i_q", "i_d"], "elektryczne"),
        ("napięcia u_q / u_d", ["u_q", "u_d"], "elektryczne"),
        ("coolant / ambient", CONTEXT_FEATURES, "kontekstowe"),
        ("temperatury", TEMPERATURE_FEATURES, "termiczne"),
    ]

    fig, axes = plt.subplots(
        len(panels),
        1,
        figsize=(13, 2.1 * len(panels)),
        sharex=True
    )

    for ax, (title, cols, _role) in zip(axes, panels):
        for c in cols:
            ax.plot(
                t_min,
                frame[c],
                lw=0.9,
                label=c
            )

        ax.set_ylabel(title, fontsize=8)
        ax.legend(
            loc="upper right",
            fontsize=7,
            ncol=len(cols)
        )

    axes[-1].set_xlabel("Czas [min]")

    dyn = float(
        profile_summary.loc[
            profile_summary[ID_COL] == pid,
            "dynamics_score"
        ].iloc[0]
    )

    fig.suptitle(
        f"Profil {pid} – {len(frame):,} próbek, "
        f"dynamics_score={dyn:.3f} | {DATA_LABEL}",
        y=1.001
    )

    fig.tight_layout()

    save_figure(
        fig,
        f"07_timeseries_profile_{pid}"
    )

    plt.show()


for pid in representative_profiles:
    plot_profile_timeseries(pid)

### 7.1 Automatyczna detekcja zdarzeń w obrębie profilu

Dla najbardziej dynamicznego profilu automatycznie identyfikujemy okresy **przyspieszania, zwalniania oraz stabilnej pracy**.

Klasyfikacja opiera się na znaku i wartości wygładzonej pochodnej `motor_speed`. Dodatkowo wyznaczamy największy zakres zmian temperatury.

Pozwala to przekształcić jakościową ocenę wykresów szeregów czasowych w wartości liczbowe.

In [ ]:
def characterise_dynamics(pid: int) -> pd.DataFrame:
    frame = grouped.get_group(pid).reset_index(drop=True)

    speed = frame["motor_speed"].to_numpy()

    win = max(
        3,
        int(5 * SAMPLING_FREQUENCY_HZ) | 1
    )  # około 5-sekundowe okno wygładzające

    kernel = np.ones(win) / win

    speed_smooth = np.convolve(
        speed,
        kernel,
        mode="same"
    )

    dspeed = (
        np.gradient(speed_smooth)
        * SAMPLING_FREQUENCY_HZ
    )  # zmiana na sekundę

    thr = 0.5 * np.std(dspeed)

    n = len(dspeed)

    accel = int((dspeed > thr).sum())
    decel = int((dspeed < -thr).sum())
    stable = n - accel - decel

    temp_mean = frame[
        TEMPERATURE_FEATURES
    ].mean(axis=1).to_numpy()

    return (
        pd.DataFrame(
            {
                "state": [
                    "przyspieszanie",
                    "zwalnianie",
                    "stabilna praca"
                ],
                "samples": [
                    accel,
                    decel,
                    stable
                ],
                "time_pct": [
                    100 * accel / n,
                    100 * decel / n,
                    100 * stable / n
                ],
            }
        ),
        float(temp_mean.max() - temp_mean.min())
    )


dyn_table, thermal_span = characterise_dynamics(
    most_variable_profile
)

print(
    f"Charakterystyka dynamiki profilu "
    f"{most_variable_profile}"
)

print("-" * 60)

print(
    dyn_table.to_string(index=False)
)

print(
    f"\nZakres termiczny "
    f"(max–min średniej temperatury) w profilu: "
    f"{thermal_span:.3f}"
)

> **Wnioski statystyczne.**  
> Każdy reprezentatywny profil można opisać udziałem czasu spędzonego w fazie przyspieszania, zwalniania i stabilnej pracy, a także zakresem zmian temperatury.
>
> **Interpretacja dziedzinowa.**
>
> - Profile przejściowe, charakteryzujące się dużym udziałem przyspieszania i zwalniania, mogą być podobne pod względem **kształtu przebiegu**, nawet jeżeli poszczególne rampy występują w nieco innych momentach. Jest to wczesna przesłanka przemawiająca za rozważeniem metod uwzględniających kształt szeregu, takich jak **DTW** lub **k-Shape**, zamiast polegania wyłącznie na odległości euklidesowej.
> - Profile stabilne koncentrują się wokół względnie stałych punktów pracy, podczas gdy kanały temperaturowe zmieniają się wolniej. Ilustruje to **bezwładność cieplną** układu: temperatura może nadal wzrastać po ustabilizowaniu się prędkości i momentu obrotowego.
>
> Zjawisko tego opóźnienia będzie szczególnie istotne podczas definiowania ograniczeń dla późniejszej augmentacji danych.

## 8. Zależności pomiędzy zmiennymi

Analizujemy, w jaki sposób poszczególne kanały zmieniają się względem siebie, wykorzystując korelację **Pearsona** oraz **Spearmana**.

Korelacja Pearsona opisuje przede wszystkim zależności liniowe, natomiast korelacja Spearmana pozwala wykrywać zależności monotoniczne na podstawie rang.

Porównanie obu współczynników może pomóc w identyfikacji zależności nieliniowych. Przykładowo para zmiennych o stosunkowo słabej korelacji Pearsona, lecz silnej korelacji Spearmana może wskazywać na zależność monotoniczną, ale nieliniową.

In [ ]:
for ax, mat, name in [
    (axes[0], pearson, "Pearson (liniowa)"),
    (axes[1], spearman, "Spearman (rangowa)")
]:
    sns.heatmap(
        mat,
        mask=mask,
        annot=True,
        fmt=".2f",
        cmap="coolwarm",
        center=0,
        vmin=-1,
        vmax=1,
        square=True,
        cbar_kws={"shrink": 0.7},
        ax=ax,
        annot_kws={"size": 7}
    )
    ax.set_title(f"{name} — korelacja [{DATA_LABEL}]")

fig.tight_layout()
save_figure(fig, "08_correlation_heatmaps")
plt.show()

### 8.1 Tabela analizy korelacji

Dla każdej unikalnej pary zmiennych zestawiane są oba współczynniki korelacji oraz miara nieliniowości:

`|Spearman| - |Pearson|`.

Tabela jest sortowana według bezwzględnej wartości współczynnika Pearsona i eksportowana do pliku `correlation_analysis.csv`.

In [ ]:
print("Najsilniejsze zależności liniowe")
print(correlation_analysis.head(8).to_string(index=False))

print("\nNajbardziej nieliniowe zależności (wysoki Spearman, relatywnie niski Pearson)")
print(
    correlation_analysis
    .sort_values("nonlinearity", ascending=False)
    .head(8)
    .to_string(index=False)
)

print("\nNajsłabsze zależności")
print(
    correlation_analysis
    .sort_values("abs_pearson")
    .head(6)
    .to_string(index=False)
)

### 8.2 Pary zmiennych uzasadnione fizycznie

Poniższa siatka wykresów punktowych przedstawia zależności, których występowania można oczekiwać na podstawie charakterystyki fizycznej układu: prędkość względem momentu obrotowego, obie zmienne operacyjne względem temperatury, prąd względem momentu (`i_q`), prąd osłabiania pola względem prędkości (`i_d`) oraz temperaturę chłodziwa i otoczenia względem temperatur elementów silnika.

W celu zachowania czytelności wykresów dane są podpróbkowane. Każdy wykres zawiera również wartości współczynników korelacji Pearsona `r` oraz Spearmana `ρ`.

In [ ]:
fig.suptitle(
    f"Zależności pomiędzy zmiennymi uzasadnione fizycznie [{DATA_LABEL}]",
    y=1.001
)

### 8.3 Wykres par zmiennych operacyjnych i kontekstowych

Wykres par dla potencjalnych zmiennych wejściowych do klasteryzacji (zmienne sterujące + zmienne kontekstowe) umożliwia jednoczesną ocenę ich wspólnej struktury oraz rozkładów brzegowych.

Ze względu na wysoki koszt obliczeniowy wykresów typu `pairplot` dane są silnie podpróbkowane. Analiza ta ma charakter jakościowy.

In [ ]:
pp.figure.suptitle(
    f"Wykres par — potencjalne zmienne wejściowe do klasteryzacji [{DATA_LABEL}]",
    y=1.02
)

**Wyniki statystyczne.** Plik `correlation_analysis.csv` szereguje wszystkie pary zmiennych według siły zależności liniowej oraz wskazuje potencjalne zależności nieliniowe. Siatka wykresów punktowych przedstawia wartości `r` oraz `ρ` dla każdej analizowanej pary.

**Interpretacja dziedzinowa.**

- Zależność `i_q ↔ torque` powinna należeć do najsilniejszych — prąd w osi q jest bezpośrednio związany z generowanym momentem obrotowym. Jeżeli analiza to potwierdzi, zmienne te są w dużym stopniu redundantne i nie powinny jednocześnie dominować w zbiorze cech używanym do klasteryzacji.
- Zależność `motor_speed ↔ u_q/u_d` powinna być silna. Przy dużej prędkości `i_d` może przyjmować coraz bardziej ujemne wartości w wyniku osłabiania pola (field weakening).
- Kanały temperaturowe są ze sobą silnie skorelowane i reagują na zmianę punktu pracy z opóźnieniem. Są więc zmiennymi wynikowymi, a nie niezależnymi czynnikami sterującymi.
- Para o wysokiej korelacji Spearmana, ale niskiej Pearsona (np. prąd → temperatura przez nasycającą się ścieżkę cieplną) wskazuje na zależność nieliniową i pokazuje ograniczenia selekcji cech opartej wyłącznie na korelacji liniowej.

## 9. Analiza zachowania termicznego

Cztery kanały temperatury — `pm`, `stator_yoke`, `stator_tooth` oraz `stator_winding` — reprezentują zmienne opisujące odpowiedź termiczną badanego układu.

W tej części porównujemy ich średnie i maksymalne wartości, szybkość nagrzewania, opóźnienie względem zmian obciążenia oraz określamy, który element osiąga najwyższą temperaturę, nagrzewa się najszybciej i wykazuje największą stabilność.

Zmienne temperaturowe nie są wykorzystywane jako wejście do klasteryzacji, ale mają kluczowe znaczenie podczas późniejszej walidacji klastrów oraz definiowania ograniczeń dla procesu augmentacji.

In [ ]:
print(f"Najgorętszy komponent       : {hottest_component}")
print(f"Najszybciej nagrzewający się: {fastest_component}")
print(f"Najbardziej stabilny komponent: {most_stable_component}")
thermal_summary

### 9.1 Przebiegi temperatury i opóźnienie cieplne

Dla reprezentatywnego profilu o najwyższej temperaturze zestawiamy cztery przebiegi temperatury z uproszczoną miarą wymuszenia cieplnego:

`torque² + 0.4 · motor_speed²`

która pełni rolę przybliżonego wskaźnika strat omowych i mechanicznych. Wszystkie przebiegi są skalowane metodą min–max wyłącznie na potrzeby wizualizacji.

Widoczne przesunięcie czasowe pomiędzy maksimum wymuszenia a maksimum poszczególnych temperatur reprezentuje opóźnienie cieplne układu.

In [ ]:
ax.plot(
    t_min,
    drive,
    color="black",
    lw=1.6,
    ls="--",
    label="wymuszenie cieplne (przybliżenie strat)"
)

ax.set(
    xlabel="czas (minuty)",
    ylabel="wartość skalowana metodą min–max",
    title=f"Przebiegi temperatury względem wymuszenia cieplnego — profil {hottest_profile} [{DATA_LABEL}]"
)

# Wyznaczenie opóźnienia za pomocą maksimum korelacji krzyżowej
# pomiędzy wymuszeniem cieplnym a każdą temperaturą.
print("Szacowane opóźnienie cieplne (wymuszenie → temperatura), profil:", hottest_profile)

### 9.2 Korelacja temperatur i struktura zależności

Temperatury poszczególnych elementów są ze sobą silnie skorelowane, jednak występują pomiędzy nimi istotne różnice czasowe wynikające z bezwładności cieplnej.

Mapa korelacji oraz wykres `stator_winding` względem `pm`, którego punkty oznaczono kolorem odpowiadającym wartości momentu obrotowego, pokazują różnice w nagrzewaniu wirnika i stojana pod wpływem obciążenia.

In [ ]:
axes[0].set_title("Korelacja temperatur")

axes[1].set(
    xlabel="temperatura uzwojenia stojana",
    ylabel="temperatura magnesu trwałego (pm)",
    title="Temperatura wirnika i stojana"
)

fig.suptitle(
    f"Struktura zależności termicznych [{DATA_LABEL}]",
    y=1.02
)

**Wyniki statystyczne.** Podsumowanie termiczne szereguje komponenty według maksymalnej temperatury, szybkości nagrzewania i zmienności. Analiza korelacji krzyżowej pozwala dodatkowo oszacować opóźnienie cieplne każdego komponentu.

**Interpretacja dziedzinowa.** Uzwojenie stojana jest zwykle najgorętszym i najszybciej nagrzewającym się elementem, co wynika z jego stosunkowo niewielkiej bezwładności cieplnej oraz bezpośredniego oddziaływania strat omowych. Jarzmo stojana reaguje wolniej i jest bardziej stabilne ze względu na większą masę cieplną.

Temperatura `pm` wirnika ma inną dynamikę i własne opóźnienie, ponieważ przepływ ciepła pomiędzy wirnikiem i stojanem odbywa się m.in. przez szczelinę powietrzną. Obserwowane opóźnienie pokazuje, że temperatury stanowią opóźnioną odpowiedź na historię punktu pracy. Ma to istotne znaczenie dla augmentacji: zmiana przebiegów prędkości lub momentu nie powinna prowadzić do niezależnej, przypadkowej modyfikacji przebiegów temperatury.

## 10. Analiza zachowania zmiennych elektrycznych

Napięcia i prądy w układzie współrzędnych d–q (`u_q`, `u_d`, `i_q`, `i_d`) należą do najszybciej zmieniających się i najbardziej bezpośrednio sterowanych sygnałów.

Analizujemy ich rozkłady, zakresy pracy, dynamikę zmian oraz wzajemne zależności. Dodatkowo identyfikujemy zmienną elektryczną charakteryzującą się największą dynamiką.

In [ ]:
# średnia bezwzględna zmiana na próbkę – miara dynamiki sygnału
# najsilniejsze zależności pomiędzy zmiennymi elektrycznymi
# a pozostałymi zmiennymi na podstawie tabeli korelacji

print(f"Najbardziej dynamiczna zmienna elektryczna: {most_dynamic_electrical}")

print("\nNajsilniejsze zależności zmiennych elektrycznych")

axes[0].set(
    xlabel="i_q (prąd związany z momentem)",
    ylabel="i_d (prąd osi d)",
    title="Płaszczyzna prądów, kolor = prędkość silnika"
)

axes[1].set(
    xlabel="u_q",
    ylabel="u_d",
    title="Płaszczyzna napięć, kolor = prędkość silnika"
)

fig.suptitle(
    f"Obszar pracy zmiennych elektrycznych [{DATA_LABEL}]",
    y=1.02
)

**Wyniki statystyczne.** Raport dla zmiennych elektrycznych szereguje cztery kanały według średniej bezwzględnej zmiany pomiędzy kolejnymi próbkami, będącej miarą dynamiki sygnału, oraz przedstawia ich zakresy pracy i najsilniejsze korelacje.

**Interpretacja dziedzinowa.** `i_q` zazwyczaj odzwierciedla zapotrzebowanie na moment i charakteryzuje się dużą dynamiką. `i_d` przyjmuje wyraźnie ujemne wartości przede wszystkim w obszarze osłabiania pola przy wysokich prędkościach.

Płaszczyzna prądów może zatem pełnić rolę mapy różnych reżimów pracy silnika. Ponieważ zmienne elektryczne są silnie ze sobą powiązane i częściowo redundantne, przestrzeń klasteryzacji nie musi zawierać wszystkich sześciu sygnałów. Odpowiednio dobrany podzbiór może wystarczająco dobrze reprezentować punkt pracy bez nadmiernego uwzględniania części elektrycznej układu.

## 11. Analiza stanów pracy

Para (`motor_speed`, `torque`) definiuje mechaniczny punkt pracy silnika. Gęstość obserwacji na tej płaszczyźnie pokazuje, które obszary pracy są często odwiedzane przez maszynę.

Obszary o wysokiej gęstości odpowiadają typowym reżimom pracy, natomiast obszary puste reprezentują warunki nieobecne w zbiorze danych.

Na tej podstawie definiujemy następnie potencjalne stany pracy (np. niskie/średnie/wysokie obciążenie, bieg jałowy, wysoka prędkość), które w kolejnych etapach mogą zostać porównane z klastrami wykrytymi przez algorytmy uczenia nienadzorowanego.

In [ ]:
ax0.set_title("Rozkład prędkości silnika")
ax1.set_title("Rozkład momentu obrotowego")

ax2.set(
    xlabel="motor_speed",
    ylabel="torque",
    title="Gęstość punktów prędkość–moment (hexbin, liczność logarytmiczna)"
)

ax3.set(
    xlabel="motor_speed",
    ylabel="torque",
    title="Gęstość punktów prędkość–moment (KDE)"
)

fig.suptitle(
    f"Analiza stanów pracy [{DATA_LABEL}]",
    y=0.995
)

### 11.1 Potencjalne definicje stanów pracy

Na podstawie progów wyznaczonych bezpośrednio z danych (tercyli prędkości i momentu obrotowego dla całego zbioru) każdej obserwacji przypisujemy potencjalny stan pracy.

Następnie określamy, jaki udział czasu zbiór danych spędza w poszczególnych stanach.

Definicje te należy traktować jako hipotezy, które zostaną później porównane z klastrami wykrytymi przez algorytmy uczenia nienadzorowanego, a nie jako rzeczywiste etykiety klas.